# Hyperparameter Search: Grid, Random & Optuna

> A self-contained companion to **Day 2**. In the main notebook we saw a quick taste of grid search and
> Optuna; here we go deeper and compare the three workhorse strategies head-to-head on MNIST:
>
> 1. **Grid search** — try every combination on a fixed grid
> 2. **Random search** — sample combinations at random (often surprisingly strong!)
> 3. **Optuna** — Bayesian/TPE search that learns from past trials, plus *pruning* and *importance*
>
> Everything runs on a small MNIST subset so each search finishes in a minute or two. The whole point
> is to experiment: change the ranges, the budget, the model, and watch what happens.

In [ ]:
import time
import itertools
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset, DataLoader
import matplotlib.pyplot as plt

# A hyperparameter SEARCH trains many tiny models back-to-back. For models this small the CPU is
# typically as fast as a GPU (per-fit data-transfer overhead cancels the speed-up) and keeps the
# search loop simple and rock-stable, so we deliberately use the CPU here. If you scale this up to
# larger models, switch to 'mps' (Apple Silicon) or 'cuda'.
device = torch.device('cpu')
print('Using device:', device)

# Small MNIST slices so every search is fast (this is itself a tuning best-practice!)
transform = transforms.ToTensor()
full = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(Subset(full, range(0, 8000)),  batch_size=128, shuffle=True)
val_loader   = DataLoader(Subset(full, range(8000, 10000)), batch_size=128, shuffle=False)


class MLP(nn.Module):
    """A small configurable MLP: one hidden layer of `n_units`, with optional dropout."""
    def __init__(self, n_units=128, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, n_units), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(n_units, 10),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(lr, n_units, dropout, epochs=4):
    """Train one configuration and return its final validation loss (lower = better)."""
    torch.manual_seed(0)
    model = MLP(n_units, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            crit(model(xb), yb).backward()
            opt.step(); opt.zero_grad()
    model.eval()
    with torch.no_grad():
        losses = [crit(model(xb.to(device)), yb.to(device)).item() for xb, yb in val_loader]
    return float(np.mean(losses))

## 1. Grid Search

Pick a few values per hyperparameter and try **every** combination. Below: 3 learning rates × 3 hidden
sizes × 2 dropout rates = **18** configurations. Note how the run count is the *product* of the options —
add one more hyperparameter and it multiplies again (the *curse of dimensionality*).

In [ ]:
grid = {
    'lr':      [1e-2, 1e-3, 1e-4],
    'n_units': [64, 128, 256],
    'dropout': [0.0, 0.3],
}
combos = list(itertools.product(grid['lr'], grid['n_units'], grid['dropout']))
print(f'Grid size: {len(combos)} configurations\n')

t0 = time.time()
grid_results = []
for lr, n_units, dropout in combos:
    val = evaluate(lr, n_units, dropout)
    grid_results.append({'lr': lr, 'n_units': n_units, 'dropout': dropout, 'val': val})
grid_time = time.time() - t0

best = min(grid_results, key=lambda r: r['val'])
print(f'Grid search: {len(combos)} runs in {grid_time:.0f}s')
print('Best:', best)

## 2. Random Search

A famous result (Bergstra & Bengio, 2012) is that for the *same compute budget*, **random search often
beats grid search** — because usually only a couple of hyperparameters really matter, and random
sampling explores more distinct values of those, instead of wasting runs on a rigid lattice.

Let's give random search the **same budget** (18 runs) and compare.

In [ ]:
rng = np.random.default_rng(0)
budget = len(combos)

t0 = time.time()
random_results = []
for _ in range(budget):
    lr = float(10 ** rng.uniform(-4, -2))        # log-uniform between 1e-4 and 1e-2
    n_units = int(rng.choice([64, 128, 256]))
    dropout = float(rng.uniform(0.0, 0.5))
    val = evaluate(lr, n_units, dropout)
    random_results.append({'lr': lr, 'n_units': n_units, 'dropout': dropout, 'val': val})
random_time = time.time() - t0

best_random = min(random_results, key=lambda r: r['val'])
print(f'Random search: {budget} runs in {random_time:.0f}s')
print('Best:', {k: (round(v, 5) if isinstance(v, float) else v) for k, v in best_random.items()})
print(f'\nGrid   best val loss: {best["val"]:.4f}')
print(f'Random best val loss: {best_random["val"]:.4f}')

## 3. Optuna — Search That Learns

Grid and random search are *non-adaptive*: every configuration is chosen without looking at the results
so far. **Optuna** instead uses past trials to decide what to try next (its default sampler, **TPE**, is
a form of Bayesian optimisation). You write an `objective` that returns a score; Optuna does the rest.

In [ ]:
# pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    n_units = trial.suggest_categorical('n_units', [64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    return evaluate(lr, n_units, dropout)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=0))
t0 = time.time()
study.optimize(objective, n_trials=budget)        # same budget again, for a fair comparison
optuna_time = time.time() - t0

print(f'Optuna: {budget} trials in {optuna_time:.0f}s')
print('Best params:', study.best_params)
print(f'Best val loss: {study.best_value:.4f}')

# best-so-far curve: watch Optuna home in
vals = [t.value for t in study.trials]
best_so_far = np.minimum.accumulate(vals)
plt.plot(range(1, len(vals) + 1), vals, 'o', alpha=0.4, label='trial value')
plt.plot(range(1, len(vals) + 1), best_so_far, '-', label='best so far')
plt.xlabel('trial'); plt.ylabel('validation loss'); plt.legend()
plt.title('Optuna optimisation history')
plt.show()

## 4. Pruning: Don't Finish Hopeless Trials

A big efficiency win: if a configuration is clearly doing badly after one epoch, why train it for four?
**Pruning** lets Optuna stop unpromising trials early. The objective reports its score after each epoch,
and a *pruner* decides whether to give up. This lets you afford many more trials for the same compute.

In [ ]:
def objective_pruning(trial):
    lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    n_units = trial.suggest_categorical('n_units', [64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.0, 0.5)

    torch.manual_seed(0)
    model = MLP(n_units, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for epoch in range(6):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            crit(model(xb), yb).backward()
            opt.step(); opt.zero_grad()

        model.eval()
        with torch.no_grad():
            val = float(np.mean([crit(model(xb.to(device)), yb.to(device)).item()
                                 for xb, yb in val_loader]))
        trial.report(val, epoch)                 # tell Optuna how this trial is doing
        if trial.should_prune():                 # ...and let it pull the plug early
            raise optuna.TrialPruned()
    return val

pruned_study = optuna.create_study(direction='minimize',
                                   sampler=optuna.samplers.TPESampler(seed=0),
                                   pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
pruned_study.optimize(objective_pruning, n_trials=25)

n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in pruned_study.trials)
print(f'Ran 25 trials; {n_pruned} were pruned early (saving training time).')
print('Best params:', pruned_study.best_params, '-> val loss', round(pruned_study.best_value, 4))

## 5. Which Hyperparameters Actually Mattered?

Optuna can estimate how much each hyperparameter influenced the score. This tells you where to focus:
widen the range of the *important* ones, and stop wasting search budget on the ones that barely move the
needle.

In [ ]:
importances = optuna.importance.get_param_importances(study)
print('Hyperparameter importance (fraction of variance explained):')
for name, imp in importances.items():
    print(f'  {name:10s} {imp:.2f}')

plt.barh(list(importances.keys()), list(importances.values()))
plt.xlabel('importance'); plt.title('Which hyperparameters mattered most?')
plt.gca().invert_yaxis()
plt.show()

## Where to Take This

- **More hyperparameters.** Add `batch_size`, number of layers, optimiser choice, weight decay — Optuna
  handles high-dimensional spaces gracefully where grids cannot.
- **Bigger budget, real model.** Drop these search functions around the CNN (Day 3) or LSTM (Day 4) and
  tune those instead — the `objective` pattern is identical.
- **Persistence & dashboards.** `optuna.create_study(storage=...)` saves trials to a database so a search
  can be paused, resumed, and even run in parallel; `optuna-dashboard` visualises it live.
- **Smarter pruners & samplers.** Explore `HyperbandPruner` and `CmaEsSampler` for larger searches.

The key takeaway: **automated search turns tuning from guesswork into a budgeted experiment.** Decide how
many trials you can afford, define sensible ranges, and let the search spend that budget wisely.